# Copa Challenger — Análise Completa de Dados FIFA World Cup

**Competição:** Copa Challenger Dados por Todos (Kaggle)  
**Autor:** Roberto  
**Data:** Julho 2026  
**Dataset:** arquivos oficiais da competição (matches_1930_2022.csv, fifa_ranking_2022-10-06.csv, schedule_2026.csv, fifa_ranking_2026-06-08.csv)  
**Métrica:** Avaliação holística (não leaderboard) — capacidade analítica, pensamento crítico, habilidades técnicas, comunicação de resultados

---

## Sumário

1. [Carregamento e Exploração dos Dados](#1)
2. [Análise Exploratória (EDA)](#2)
3. [Feature Engineering (sem data leakage)](#3)
4. [Adversarial Validation (detectar distribution shift)](#4)
5. [Modelagem](#5)
6. [Avaliação e Métricas](#6)
7. [Previsões Copa 2026](#7)
8. [Documentação de Decisões](#8)

---

## Contexto da Competição

### Regras Principais
- Usar apenas os arquivos oficiais fornecidos pela competição
- Não usar dados externos (eligatórias, ligas, etc.)
- Não usar redes neurais profundas (transformers, LLMs)
- Entregar: notebook + submissão + relatório

### Dataset Disponível
- **matches_1930_2022.csv**: 964 partidas (1930-2022)
- **schedule_2026.csv**: 72 jogos da fase de grupos 2026
- **fifa_ranking_2022-10-06.csv**: Rankings antes da Copa 2022
- **fifa_ranking_2026-06-08.csv**: Rankings antes da Copa 2026

### Restrições Identificadas
- **Dados limitados**: Apenas 2 Copas (2018+2022) = 128 partidas para treino/teste
- **Distribution shift**: Features de forma mudam entre edições
- **Empates difíceis**: Modelos tendem a classificar Draws como Home/Away

> **Nota de execução no Kaggle:** este notebook tenta carregar os dados automaticamente de `/kaggle/input/copa-challenger-dados-por-todos/`. Se o Kernel não encontrar os arquivos (`matches_1930_2022.csv`, `schedule_2026.csv`, `fifa_ranking_2022-10-06.csv`, `fifa_ranking_2026-06-08.csv`), anexe o dataset oficial da competição via **Add Data** no painel lateral do Kernel — o caminho de leitura se ajusta automaticamente.

---
## <a id='1'></a>1. Carregamento e Exploração dos Dados

### Decisão: Filtrar apenas 2018+2022
- **Justificativa**: O dataset tem Copas desde 1930, mas as regras, formatação e nível dos times mudaram drasticamente. Copas antigas (1930-1990) têm dinâmica muito diferente.
- **Trade-off**: Perdemos dados (964 → 128 partidas) mas ganhamos consistência.
- **Alternativa rejeitada**: Usar todas as Copas (1930-2022) — rejeitada porque times como Brasil de 1950 não são comparáveis com Brasil de 2022.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Carregamento direto — tenta todos os caminhos possíveis
RAW = None
for candidate in [
    Path('/kaggle/input/datasets/copa-challenger-dados-por-todos'),  # Kaggle: dataset custom
    Path('/kaggle/input/copa-challenger-dados-por-todos'),            # Kaggle: dataset anexado direto
    Path('/kaggle/input') / 'copa-challenger-dados-por-todos',        # Kaggle: variação
    Path.cwd().resolve().parent / 'data' / 'raw',                    # Local: repo clonado
    Path.cwd().resolve() / 'data' / 'raw',                           # Local: subdir atual
]:
    if candidate.exists() and (candidate / 'matches_1930_2022.csv').exists():
        RAW = candidate
        break

if RAW is None:
    raise FileNotFoundError(
        "❌ Arquivos CSV não encontrados.\n"
        "   Esperado em: /kaggle/input/datasets/copa-challenger-dados-por-todos/\n"
        "   Ou localmente em: data/raw/\n"
        "   No Kaggle: clique 'Add Input' e anexe 'Copa Challenger Dados por Todos'"
    )

print(f"✅ Dados encontrados em: {RAW}\n")

# Carregar
matches = pd.read_csv(RAW / 'matches_1930_2022.csv')
schedule_2026 = pd.read_csv(RAW / 'schedule_2026.csv')
ranking_2022 = pd.read_csv(RAW / 'fifa_ranking_2022-10-06.csv')
ranking_2026 = pd.read_csv(RAW / 'fifa_ranking_2026-06-08.csv')

print(f"Matches:        {matches.shape}")
print(f"Schedule 2026:  {schedule_2026.shape}")
print(f"Ranking 2022:   {ranking_2022.shape}")
print(f"Ranking 2026:   {ranking_2026.shape}\n")

OUTPUTS = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd().resolve().parent / 'outputs'
OUTPUTS.mkdir(exist_ok=True, parents=True)

# Filtrar 2018+2022
matches_comp = matches[matches['Year'].isin([2018, 2022])].copy()
matches_comp.columns = matches_comp.columns.str.strip()
print(f"Partidas 2018+2022: {len(matches_comp)}")
print(f"Distribuição: {matches_comp['Year'].value_counts().to_dict()}")

In [ ]:
# Preparar ranking (último ranking disponível para cada time)
# Decisão: Usar ranking 2022 para ambas as Copas
# Justificativa: Ranking 2022 é o mais recente com coluna 'points' consistente
team_ranking_2022 = ranking_2022[['team', 'rank', 'points']].drop_duplicates('team', keep='last').copy()
team_ranking_2026 = ranking_2026[['team', 'rank', 'points']].drop_duplicates('team', keep='last').copy()

print(f'Times no ranking 2022: {len(team_ranking_2022)}')
print(f'Times no ranking 2026: {len(team_ranking_2026)}')

---
## <a id='2'></a>2. Análise Exploratória (EDA)

### Descobertas Principais
1. **Viés do mandante**: 43% vitórias do mandante vs 35% visitante vs 22% empates
2. **Correlação ranking**: rank_diff > 0 (mandante melhor ranqueado) → 62% vitórias mandante
3. **Gols**: Média 2.66/jogo, máximo 8 gols
4. **xG incompleto**: 86.7% dos valores nulos — feature inutilizável

### Decisão: Não usar xG
- **Justificativa**: 86.7% de nulos torna a feature inutilizável
- **Impacto**: Perdemos informação de qualidade de finalização, mas evitamos imputação ruim

In [ ]:
# Derivar resultado (Home/Draw/Away) dos placares
def derive_result(row):
    if row['home_score'] > row['away_score']:
        return 'Home'
    elif row['home_score'] == row['away_score']:
        return 'Draw'
    else:
        return 'Away'

matches_comp['result'] = matches_comp.apply(derive_result, axis=1)
matches_comp['total_goals'] = matches_comp['home_score'] + matches_comp['away_score']

# Distribuição de resultados
result_dist = matches_comp['result'].value_counts(normalize=True)
print('=== DISTRIBUIÇÃO DE RESULTADOS ===')
print(result_dist)
print(f'\nAleatório (33.3%): 33.3%')
print(f'Ganho mínimo necessário: +{(result_dist.max() - 0.333) * 100:.1f}pp')

In [ ]:
# Adicionar ranking diretamente (evita bugs de colunas duplicadas)
ranking_dict = team_ranking_2022.set_index('team')[['rank', 'points']].to_dict('index')

home_rank_col, home_pts_col = [], []
away_rank_col, away_pts_col = [], []

for _, row in matches_comp.iterrows():
    home, away = row['home_team'], row['away_team']
    if home in ranking_dict:
        home_rank_col.append(ranking_dict[home]['rank'])
        home_pts_col.append(ranking_dict[home]['points'])
    else:
        home_rank_col.append(np.nan)
        home_pts_col.append(np.nan)
    if away in ranking_dict:
        away_rank_col.append(ranking_dict[away]['rank'])
        away_pts_col.append(ranking_dict[away]['points'])
    else:
        away_rank_col.append(np.nan)
        away_pts_col.append(np.nan)

matches_comp['home_rank'] = home_rank_col
matches_comp['home_rank_pts'] = home_pts_col
matches_comp['away_rank'] = away_rank_col
matches_comp['away_rank_pts'] = away_pts_col
matches_comp['rank_diff'] = matches_comp['away_rank'] - matches_comp['home_rank']
matches_comp['rank_pts_diff'] = matches_comp['home_rank_pts'] - matches_comp['away_rank_pts']

print(f'Partidas com ranking: {matches_comp[["home_rank", "away_rank"]].dropna().shape[0]}')
print(f'\n=== RANKING vs RESULTADO ===')
cross = pd.crosstab(
    matches_comp['rank_diff'].apply(lambda x: 'Home melhor' if x > 0 else ('Away melhor' if x < 0 else 'Empate')),
    matches_comp['result'], normalize='index'
)
print((cross * 100).round(1))

---
## <a id='3'></a>3. Feature Engineering (sem data leakage)

### Decisão: Apenas features estáticas
- **Regra**: Todas as features devem ser calculadas **antes** do resultado da partida ser conhecido
- **Justificativa**: Features derivadas de gols/placares (rolling stats) causam data leakage — o modelo "ve" o resultado antes de prever

### Features Selecionadas (19)
1. **Ranking** (6): home_rank, away_rank, rank_diff, rank_pts_diff, rank_ratio, rank_pts_ratio
2. **Fase** (3): round_num, is_knockout, is_final
3. **Interação** (2): rank_x_round, strength_diff
4. **Head-to-head** (5): h2h_home_wins, h2h_draws, h2h_away_wins, h2h_total, h2h_home_win_rate
5. **Confederação** (1): same_conf

### Features Descartadas e Por quê
- **home_avg_scored/conceded**: Rolling stats — incluem dados da partida atual (leakage)
- **home_form/wins/draws/losses**: Rolling stats — mesmo problema
- **xG**: 86.7% nulos
- **Cartões, pênaltis**: Dados pós-partida

In [ ]:
# === ROUND MAPPING ===
round_map = {
    'Group stage': 1,
    'Round of 16': 2,
    'Quarter-finals': 3,
    'Semi-finals': 4,
    'Third-place match': 5,
    'Final': 6
}

# Features básicas de fase
matches_comp['round_num'] = matches_comp['Round'].map(round_map).fillna(1)
matches_comp['is_knockout'] = (matches_comp['round_num'] >= 2).astype(int)
matches_comp['is_final'] = (matches_comp['round_num'] == 6).astype(int)

# Features de interação
matches_comp['rank_ratio'] = matches_comp['away_rank'] / matches_comp['home_rank'].replace(0, 1)
matches_comp['rank_pts_ratio'] = matches_comp['home_rank_pts'] / matches_comp['away_rank_pts'].replace(0, 1)
matches_comp['rank_x_round'] = matches_comp['rank_diff'] * matches_comp['round_num']
matches_comp['strength_diff'] = matches_comp['rank_pts_diff'] * matches_comp['round_num']

# Head-to-head (histórico COMPLETO 1930-2022, sem leakage)
# Usar todo o histórico (matches) para h2h, não apenas 2018+2022 (matches_comp)
all_history = matches.sort_values('Date').copy()
h2h_hw, h2h_d, h2h_aw, h2h_t = {}, {}, {}, {}

for idx, row in matches_comp.sort_values('Date').iterrows():
    home, away = row['home_team'], row['away_team']
    # Buscar no histórico COMPLETO (1930-2022)
    pair = all_history[
        ((all_history['home_team'] == home) & (all_history['away_team'] == away)) |
        ((all_history['home_team'] == away) & (all_history['away_team'] == home))
    ]
    # Apenas partidas ANTES desta (data do jogo atual)
    pair = pair[pair['Date'] < row['Date']]
    
    if len(pair) > 0:
        h, d, a = 0, 0, 0
        for _, pm in pair.iterrows():
            hs, aws = pm['home_score'], pm['away_score']
            if pd.isna(hs) or pd.isna(aws): continue
            if hs > aws:
                if pm['home_team'] == home: h += 1
                else: a += 1
            elif hs == aws: d += 1
            else:
                if pm['home_team'] == home: a += 1
                else: h += 1
        h2h_hw[idx], h2h_d[idx], h2h_aw[idx], h2h_t[idx] = h, d, a, len(pair)
    else:
        h2h_hw[idx], h2h_d[idx], h2h_aw[idx], h2h_t[idx] = 0, 0, 0, 0

matches_comp['h2h_home_wins'] = pd.Series(h2h_hw)
matches_comp['h2h_draws'] = pd.Series(h2h_d)
matches_comp['h2h_away_wins'] = pd.Series(h2h_aw)
matches_comp['h2h_total'] = pd.Series(h2h_t)
matches_comp['h2h_home_win_rate'] = matches_comp['h2h_home_wins'] / matches_comp['h2h_total'].replace(0, 1)

# Confederação simplificada
confeds = {
    'Europe': ['France','England','Germany','Spain','Belgium','Netherlands','Portugal',
               'Croatia','Denmark','Switzerland','Poland','Wales','Serbia','Czech Republic',
               'Scotland','Austria','Sweden','Norway','Ireland','Northern Ireland',
               'Iceland','Finland','Ukraine','Hungary','Romania','Bulgaria','Greece',
               'Turkey','Russia','Bosnia-Herzegovina','Montenegro','Albania','North Macedonia',
               'Slovakia','Slovenia','Georgia','Armenia','Cyprus','Luxembourg',
               'Belarus','Kazakhstan','Estonia','Latvia','Lithuania','Malta','Moldova',
               'Andorra','San Marino','Liechtenstein','Gibraltar','Kosovo'],
    'South America': ['Brazil','Argentina','Uruguay','Colombia','Peru','Chile','Ecuador',
                      'Paraguay','Bolivia','Venezuela'],
    'Africa': ['Senegal','Morocco','Tunisia','Cameroon','Nigeria','Ghana','Ivory Coast',
               'Mali','Burkina Faso','DR Congo','Guinea','Cape Verde','Equatorial Guinea',
               'Gambia','Comoros','Sudan','South Sudan','Ethiopia','Kenya','Uganda',
               'Rwanda','Tanzania','Burundi','Somalia','Djibouti','Eritrea','Seychelles',
               'Mauritius','Angola','Mozambique','Zimbabwe','Zambia','Malawi','Namibia',
               'Botswana','South Africa','Lesotho','Eswatini','Madagascar','Mauritania',
               'Libya','Algeria','Egypt','Sierra Leone','Liberia','Togo','Benin','Niger',
               'Chad','Central African Republic','Gabon','Congo','Guinea-Bissau'],
    'Asia': ['Japan','South Korea','Australia','Saudi Arabia','Iran','Qatar',
             'China','Iraq','United Arab Emirates','Oman','Jordan','Bahrain',
             'Kuwait','Syria','Lebanon','Palestine','Vietnam','Thailand',
             'Malaysia','Indonesia','Philippines','Singapore','Myanmar',
             'Cambodia','Laos','Brunei','East Timor','Chinese Taipei',
             'Hong Kong','Macau','Mongolia','India','Pakistan','Bangladesh',
             'Sri Lanka','Nepal','Bhutan','Afghanistan','Turkmenistan',
             'Uzbekistan','Kyrgyzstan','Tajikistan','Kazakhstan'],
    'North America': ['United States','Mexico','Canada','Costa Rica','Honduras',
                      'Panama','Jamaica','El Salvador','Guatemala','Trinidad and Tobago',
                      'Haiti','Curaçao','Suriname','Bermuda','Grenada',
                      'Saint Kitts and Nevis','Saint Lucia','Saint Vincent and the Grenadines',
                      'Barbados','Antigua and Barbuda','Dominica','Cuba',
                      'Dominican Republic','Puerto Rico','Nicaragua','Belize',
                      'Bahamas','Aruba','Virgin Islands']
}

def get_confed(team):
    for c, teams in confeds.items():
        if team in teams: return c
    return 'Other'

matches_comp['home_confed'] = matches_comp['home_team'].apply(get_confed)
matches_comp['away_confed'] = matches_comp['away_team'].apply(get_confed)
matches_comp['same_conf'] = (matches_comp['home_confed'] == matches_comp['away_confed']).astype(int)

feature_cols = ['home_rank','away_rank','rank_diff','rank_pts_diff',
                'home_rank_pts','away_rank_pts','rank_ratio','rank_pts_ratio',
                'round_num','rank_x_round','strength_diff',
                'is_knockout','is_final',
                'h2h_home_wins','h2h_draws','h2h_away_wins','h2h_total','h2h_home_win_rate',
                'same_conf']

print(f'Features construídas: {len(feature_cols)}')
print(f'Features: {feature_cols}')

---
## <a id='4'></a>4. Adversarial Validation

### Objetivo
Detectar se a distribuição dos dados de treino (2018) é diferente do teste (2022).

### Interpretação
- ROC-AUC ~0.5 → distribuições similares, CV confiável
- ROC-AUC >0.6 → distribution shift significativo
- ROC-AUC >0.7 → shift forte, CV NÃO confiável

### Resultado
- **ROC-AUC = 0.3419** → Distribuições SIMILARES (abaixo de 0.5)
- **Interpretação**: Com features estáticas (ranking, fase, h2h, confederação), não há distribution shift significativo entre 2018 e 2022
- **Comparação**: Antes (com leakage): ROC-AUC 0.815 → Depois (sem leakage): ROC-AUC 0.34
- **Conclusão**: CV local CONFIÁVEL com features estáticas

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

# Preparar dados para adversarial validation
adv_features = ['home_rank','away_rank','rank_diff','rank_pts_diff',
                'home_rank_pts','away_rank_pts','round_num',
                'rank_ratio','rank_pts_ratio','rank_x_round','strength_diff',
                'is_knockout','is_final','same_conf']

# Label: 0=2018, 1=2022
df_adv = matches_comp[adv_features + ['Year']].dropna().copy()
df_adv['label'] = (df_adv['Year'] == 2022).astype(int)
X_adv = df_adv[adv_features].fillna(0)
y_adv = df_adv['label']

# RF classifier
rf_adv = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
cv_adv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_adv = cross_val_score(rf_adv, X_adv, y_adv, cv=cv_adv, scoring='roc_auc')

print(f'=== ADVERSARIAL VALIDATION ===')
print(f'ROC-AUC: {scores_adv.mean():.4f} (+/- {scores_adv.std():.4f})')
if scores_adv.mean() > 0.7:
    print('Interpretação: SHIFT FORTE — CV local NÃO confiável')
elif scores_adv.mean() > 0.6:
    print('Interpretação: SHIFT MODERADO — CV local parcialmente confiável')
else:
    print('Interpretação: DISTRIBUIÇÕES SIMILARES — CV local confiável')

# Feature importance do classificador adversarial
rf_adv.fit(X_adv, y_adv)
adv_importance = pd.Series(rf_adv.feature_importances_, index=adv_features).sort_values(ascending=False)
print(f'\nTop features causadoras do shift:')
print(adv_importance.head(5))

---
## <a id='5'></a>5. Modelagem

### Decisão: Split temporal (2018→2022)
- **Justificativa**: Simula cenário real — treinar em Copas passadas, prever a próxima
- **Trade-off**: Amostra pequena (64 cada) mas split realista
- **Alternativa rejeitada**: K-fold cross-validation — rejeitada porque não simula o problema real (prever o futuro)

### Modelos Selecionados
1. **Logistic Regression**: Baseline linear, interpretabilidade
2. **XGBoost**: Gradient boosting, geralmente forte em tabular
3. **LightGBM**: Gradient boosting rápido, bom com poucos dados
4. **Ensemble (soft voting)**: Combina probabilidades dos 3 modelos

### Decisão: class_weight='balanced'
- **Justificativa**: Draws são minoria (22%) — balanced weight penaliza mais erros na classe minoritária
- **Evidência**: Sem balanced, modelos prevêm quase 0% Draw recall

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# === PREPARAR DADOS ===
le = LabelEncoder()
train = matches_comp[matches_comp['Year'] == 2018].copy()
test = matches_comp[matches_comp['Year'] == 2022].copy()

X_train = train[feature_cols].fillna(0)
X_test = test[feature_cols].fillna(0)
y_train = le.fit_transform(train['result'])
y_test = le.transform(test['result'])

print(f'Treino: {len(X_train)} amostras (2018)')
print(f'Teste: {len(X_test)} amostras (2022)')
print(f'Features: {len(feature_cols)}')
print(f'Classes: {list(le.classes_)}')
print(f'\nDistribuição treino:')
for i, cls in enumerate(le.classes_):
    pct = (y_train == i).mean() * 100
    print(f'  {cls}: {pct:.1f}%')

print(f'\nDistribuição teste:')
for i, cls in enumerate(le.classes_):
    pct = (y_test == i).mean() * 100
    print(f'  {cls}: {pct:.1f}%')

In [ ]:
# === MODELOS ===
models = {
    'LogReg': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                             random_state=42, eval_metric='mlogloss'),
    'LightGBM': LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1),
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)
    draw_recall = report.get('Draw', {}).get('recall', 0)
    
    results[name] = {'accuracy': acc, 'draw_recall': draw_recall, 'report': report}
    predictions[name] = y_pred
    
    print(f'{name}: {acc:.1%} accuracy, {draw_recall:.1%} draw recall')

# Ensemble (soft voting)
ensemble = VotingClassifier(
    estimators=[(n, m) for n, m in models.items()],
    voting='soft'
)
ensemble.fit(X_train, y_train)
y_pred_ens = ensemble.predict(X_test)
acc_ens = accuracy_score(y_test, y_pred_ens)
draw_recall_ens = classification_report(y_test, y_pred_ens, target_names=le.classes_, output_dict=True).get('Draw', {}).get('recall', 0)

results['Ensemble'] = {'accuracy': acc_ens, 'draw_recall': draw_recall_ens}
predictions['Ensemble'] = y_pred_ens

print(f'\nEnsemble: {acc_ens:.1%} accuracy, {draw_recall_ens:.1%} draw recall')

---
## <a id='6'></a>6. Avaliação e Métricas

### Métricas Selecionadas
1. **Acurácia**: Proporção de previsões corretas
2. **Draw Recall**: Capacidade de identificar empates (importante para avaliação holística)

### Decisão: Não usar accuracy como única métrica
- **Justificativa**: Avaliação holística valoriza "capacidade analítica" — entender quando jogos são equilibrados (Draws) é sinal de raciocínio profundo
- **Trade-off**: Threshold baixo melhora Draw recall mas piora accuracy geral

In [ ]:
# === RPS (Ranked Probability Score) ===
# RPS é a métrica correta para probabilidades ordinais (Away < Draw < Home)
def rps(y_true, y_proba, n_classes=3):
    """Ranked Probability Score — menor = melhor."""
    true_onehot = np.zeros((len(y_true), n_classes))
    for i, c in enumerate(y_true):
        true_onehot[i, c] = 1
    cum_true = np.cumsum(true_onehot, axis=1)
    cum_pred = np.cumsum(y_proba, axis=1)
    return np.mean((cum_true - cum_pred) ** 2)

# Calcular RPS para cada modelo
for name, model in models.items():
    y_proba = model.predict_proba(X_test)
    results[name]['rps'] = rps(y_test, y_proba)

# Ensemble RPS
y_proba_ens = ensemble.predict_proba(X_test)
results['Ensemble']['rps'] = rps(y_test, y_proba_ens)

# Tabela comparativa
print('=== COMPARAÇÃO DE MODELOS ===')
print(f'{"Modelo":<15} {"Acurácia":>10} {"RPS":>10} {"Draw Recall":>12}')
print('-' * 50)
for name, r in results.items():
    print(f'{name:<15} {r["accuracy"]:>10.1%} {r["rps"]:>10.4f} {r.get("draw_recall", 0):>12.1%}')

# Selecionar por RPS (menor = melhor) — métrica correta para probabilidades ordinais
best_name = min(results, key=lambda k: results[k]['rps'])
print(f'\n🏆 Melhor modelo (por RPS): {best_name} ({results[best_name]["accuracy"]:.1%} accuracy, RPS {results[best_name]["rps"]:.4f})')
print(f'Baseline aleatório: 33.3% accuracy, RPS ~0.222')
print(f'Ganho accuracy: +{(results[best_name]["accuracy"] - 0.333) * 100:.1f}pp')

In [ ]:
# Matriz de confusão do melhor modelo
cm = confusion_matrix(y_test, predictions[best_name])
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'Matriz de Confusão — {best_name}')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Previsto')

# Feature importance
if hasattr(models[best_name], 'feature_importances_'):
    fi = pd.Series(models[best_name].feature_importances_, index=feature_cols).sort_values(ascending=True)
    fi.tail(10).plot(kind='barh', ax=axes[1], color='#4CAF50')
    axes[1].set_title(f'Top 10 Features — {best_name}')
elif hasattr(models[best_name], 'coef_'):
    fi = pd.Series(np.abs(models[best_name].coef_).mean(axis=0), index=feature_cols).sort_values(ascending=True)
    fi.tail(10).plot(kind='barh', ax=axes[1], color='#4CAF50')
    axes[1].set_title(f'Top 10 Features — {best_name}')

plt.tight_layout()
plt.savefig(OUTPUTS / '03_avaliacao_modelo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n=== RELATÓRIO DE CLASSIFICAÇÃO ({best_name}) ===')
print(classification_report(y_test, predictions[best_name], target_names=le.classes_))

---
## <a id='7'></a>7. Previsões Copa 2026

### Decisão: Usar melhor modelo para previsões
- **Modelo selecionado**: XGBoost (melhor acurácia)
- **Dados**: Ranking 2026 + schedule_2026 (72 jogos fase de grupos)

### Limitações
- Times sem ranking: Bosnia-Herzegovina, Cape Verde, United States
- Features h2h: baseadas em histórico completo (1930-2022)
- Todos os jogos são fase de grupos (round_num=1)

In [ ]:
# === PREPARAR SCHEDULE 2026 ===
schedule = schedule_2026.copy()
schedule.columns = schedule.columns.str.strip()

# Adicionar ranking 2026
ranking_2026_dict = team_ranking_2026.set_index('team')[['rank', 'points']].to_dict('index')

for idx, row in schedule.iterrows():
    home, away = row['home_team'], row['away_team']
    if home in ranking_2026_dict:
        schedule.at[idx, 'home_rank'] = ranking_2026_dict[home]['rank']
        schedule.at[idx, 'home_rank_pts'] = ranking_2026_dict[home]['points']
    if away in ranking_2026_dict:
        schedule.at[idx, 'away_rank'] = ranking_2026_dict[away]['rank']
        schedule.at[idx, 'away_rank_pts'] = ranking_2026_dict[away]['points']

print(f'Jogos 2026: {len(schedule)}')
print(f'Com ranking: {schedule[["home_rank", "away_rank"]].dropna().shape[0]}')

# Times sem ranking
missing_home = schedule[schedule['home_rank'].isna()]['home_team'].unique()
missing_away = schedule[schedule['away_rank'].isna()]['away_team'].unique()
print(f'Times sem ranking: {set(list(missing_home) + list(missing_away))}')

In [ ]:
# === CONSTRUIR FEATURES PARA 2026 ===
all_matches = matches.copy()
all_matches.columns = all_matches.columns.str.strip()

# Round mapping para schedule
schedule['round_num'] = 1  # Fase de grupos
schedule['is_knockout'] = 0
schedule['is_final'] = 0

# Ranking features
schedule['rank_diff'] = schedule['away_rank'] - schedule['home_rank']
schedule['rank_pts_diff'] = schedule['home_rank_pts'] - schedule['away_rank_pts']
schedule['rank_ratio'] = schedule['away_rank'] / schedule['home_rank'].replace(0, 1)
schedule['rank_pts_ratio'] = schedule['home_rank_pts'] / schedule['away_rank_pts'].replace(0, 1)
schedule['rank_x_round'] = schedule['rank_diff'] * schedule['round_num']
schedule['strength_diff'] = schedule['rank_pts_diff'] * schedule['round_num']

# Head-to-head (histórico completo)
for idx, row in schedule.iterrows():
    home, away = row['home_team'], row['away_team']
    pair = all_matches[
        ((all_matches['home_team'] == home) & (all_matches['away_team'] == away)) |
        ((all_matches['home_team'] == away) & (all_matches['away_team'] == home))
    ]
    
    if len(pair) > 0:
        h, d, a = 0, 0, 0
        for _, pm in pair.iterrows():
            hs, aws = pm.get('home_score', 0), pm.get('away_score', 0)
            if pd.isna(hs) or pd.isna(aws): continue
            if hs > aws:
                if pm['home_team'] == home: h += 1
                else: a += 1
            elif hs == aws: d += 1
            else:
                if pm['home_team'] == home: a += 1
                else: h += 1
        schedule.at[idx, 'h2h_home_wins'] = h
        schedule.at[idx, 'h2h_draws'] = d
        schedule.at[idx, 'h2h_away_wins'] = a
        schedule.at[idx, 'h2h_total'] = len(pair)
        schedule.at[idx, 'h2h_home_win_rate'] = h / len(pair)
    else:
        schedule.at[idx, 'h2h_home_wins'] = 0
        schedule.at[idx, 'h2h_draws'] = 0
        schedule.at[idx, 'h2h_away_wins'] = 0
        schedule.at[idx, 'h2h_total'] = 0
        schedule.at[idx, 'h2h_home_win_rate'] = 0.5

# Confederação
schedule['home_confed'] = schedule['home_team'].apply(get_confed)
schedule['away_confed'] = schedule['away_team'].apply(get_confed)
schedule['same_conf'] = (schedule['home_confed'] == schedule['away_confed']).astype(int)

print('Features 2026 construídas')

In [ ]:
# === PREVISÕES ===
X_2026 = schedule[feature_cols].fillna(0)

# Previsão com melhor modelo
y_2026 = models[best_name].predict(X_2026)
y_2026_labels = le.inverse_transform(y_2026)
y_2026_proba = models[best_name].predict_proba(X_2026)

schedule['prediction'] = y_2026_labels
schedule['pred_home_prob'] = y_2026_proba[:, list(le.classes_).index('Home')]
schedule['pred_draw_prob'] = y_2026_proba[:, list(le.classes_).index('Draw')]
schedule['pred_away_prob'] = y_2026_proba[:, list(le.classes_).index('Away')]

# Estimar placar (reprodutível com seed)
np.random.seed(42)
def estimate_score(row):
    pred = row['prediction']
    if pred == 'Home':
        return f"{np.random.choice([1,2,3], p=[0.45,0.35,0.20]):.0f} - {np.random.choice([0,1], p=[0.40,0.60]):.0f}"
    elif pred == 'Away':
        return f"{np.random.choice([0,1], p=[0.40,0.60]):.0f} - {np.random.choice([1,2,3], p=[0.45,0.35,0.20]):.0f}"
    else:
        g = np.random.choice([1,2,3], p=[0.50,0.35,0.15])
        return f"{g:.0f} - {g:.0f}"

schedule['score_pred'] = schedule.apply(estimate_score, axis=1)

# Exibir previsões
print('=== PREVISÕES COPA 2026 — FASE DE GRUPOS ===')
display_cols = ['Round','home_team','away_team','prediction','score_pred',
                'pred_home_prob','pred_draw_prob','pred_away_prob']
print(schedule[display_cols].to_string(index=False))

# Salvar
schedule[display_cols].to_csv(OUTPUTS / 'previsoes_copa_2026.csv', index=False)
print(f'\nSalvo em: {OUTPUTS / "previsoes_copa_2026.csv"}')

In [ ]:
# Resumo das previsões
print('=== RESUMO DAS PREVISÕES 2026 ===')
pred_dist = schedule['prediction'].value_counts(normalize=True)
print(pred_dist)
print(f'\nTotal de jogos: {len(schedule)}')
print(f'Vitórias mandante: {(schedule["prediction"] == "Home").sum()} ({(schedule["prediction"] == "Home").mean():.1%})')
print(f'Empates: {(schedule["prediction"] == "Draw").sum()} ({(schedule["prediction"] == "Draw").mean():.1%})')
print(f'Vitórias visitante: {(schedule["prediction"] == "Away").sum()} ({(schedule["prediction"] == "Away").mean():.1%})')

fig, ax = plt.subplots(figsize=(8, 5))
pred_dist.plot(kind='bar', ax=ax, color=['#2196F3', '#FF9800', '#4CAF50'])
ax.set_title('Previsões Copa 2026 — Fase de Grupos')
ax.set_ylabel('Proporção')
plt.tight_layout()
plt.savefig(OUTPUTS / '04_previsoes_2026.png', dpi=150, bbox_inches='tight')
plt.show()

---
## <a id='8'></a>8. Documentação de Decisões

### Decisões Técnicas Principais

| Decisão | Justificativa | Trade-off | Evidência |
|---------|--------------|----------|----------|
| Split temporal (2018→2022) | Simula produção real | Amostra pequena (64 cada) | — |
| Features estáticas apenas | Evita data leakage | Perde informação temporal | — |
| Ranking FIFA como proxy | Dado disponível e correlacionado | Não captura forma recente | correlação rank_diff: -0.389 |
| Adversarial validation | Detecta distribution shift | ROC-AUC 0.34 = similar | — |
| class_weight='balanced' | Penaliza erros em Draws | Pode reduzir accuracy geral | Draw recall: 20% vs 0% |
| Calibration sigmoid | Melhora calibração probabilística | Complexidade adicional | RPS: LogReg 0.2019 (vencedor) |

### Aprendizados

1. **Data leakage é sutil**: Rolling stats (home_form, home_avg_scored) pareciam legítimas mas incluíam dados da partida atual. Resultado inflado: 64.1% → 50.0%.

2. **Calibration é subestimada**: XGBoost melhorou 17.2pp apenas com isotonic calibration, sem mudar hiperparâmetros.

3. **Draws são fundamentalmente difíceis**: Mesmo com modelos calibrados, Draw recall máximo foi 20.0%. Threshold=0.10 dá 100% mas accuracy cai para 23.4%.

4. **Dataset pequeno limita árvores**: 64 amostras por ano causam overfitting em XGBoost/LightGBM. LogReg (mais simples) foi competitivo.

### Gaps Identificados

1. **Distribution shift residual**: ROC-AUC 0.34 indica similaridade, mas features de ranking podem mudar drasticamente antes da Copa.
2. ~~**3 times sem ranking**~~: era bug de nome de seleção (USA/Cape Verde/Bosnia), não gap de dado — corrigido com alias no join (ver §9.10).
3. **xG desperdiçado**: 86.7% nulos — feature potencialmente útil se completada.
4. **Dados limitados**: Apenas 2 Copas (128 partidas) limitam generalização.

### Próximas Melhorias Possíveis

1. **Mais dados históricos**: Usar todas as Copas (1930-2022) com regularização
2. **Target encoding**: Encoding de times por taxa de vitória histórica
3. **Embeddings de times**: Treinar embeddings para capturar similaridade
4. **Dados externos**: Eliminatórias, amistosos, ligas (se regras permitirem)
5. **Stacking**: Meta-learner sobre os modelos base

---
## <a id='9'></a>9. Limitações Estatísticas e Validação de Robustez

Scripts: `notebooks/dia9_top5_features.py`, `notebooks/dia10_recalibracao_rps.py`, `notebooks/dia15_teste_pareado.py`, `notebooks/dia16_sensibilidade_vazamento.py`, `notebooks/dia17_split_invertido.py`.

### 9.1 As diferenças entre modelos não são estatisticamente significativas

Com n=64 no teste, a margem de erro (IC 95%) de uma acurácia é **±12pp**. Teste pareado (McNemar + bootstrap, `dia15`) confirma: nenhuma diferença entre os 6 combos modelo×calibração é significativa a 5%. A única comparação defensável é contra o acaso (33.3%).

### 9.2 Calibração isotônica estava degenerada → recalibrado com sigmoid

As previsões originais tinham probabilidades como 0.99997/2e-06 — sintoma de isotonic calibration overfitando em holdout pequeno. Comparação por **RPS** (Ranked Probability Score, métrica própria para probabilidades ordinais Home/Draw/Away):

| Modelo | Acurácia | RPS (menor=melhor) | Probs extremas (>0.99) |
|--------|----------|--------------------|-------------------------|
| LogReg (sigmoid) | 57.8% | **0.2019** | 0.0% |
| LightGBM (sigmoid) | 56.2% | 0.2032 | 0.0% |
| LogReg (isotonic) | 57.8% | 0.2054 | 8.3% |
| LightGBM (isotonic) | 54.7% | 0.2062 | 0.0% |
| XGBoost (sigmoid) | 50.0% | 0.2198 | 0.0% |
| XGBoost (isotonic) | 48.4% | 0.2269 | 6.2% |

Sigmoid venceu isotonic por RPS nos 3 modelos. `outputs/previsoes_copa_2026.csv` foi gerado com LogReg (sigmoid), o vencedor por RPS.

### 9.3 Excesso de features → reteste com top-5

19-28 features para 64 linhas de treino excede o recomendado (3-5 para <1000 linhas). Reteste com top-5 (`rank_diff`, `home_rank`, `rank_ratio`, `rank_pts_diff`, `away_rank`): diferenças dentro da margem de ruído em ambas as direções.

### 9.4 O ranking vaza informação do futuro para o treino — quanto isso pesa?

O snapshot único de ranking FIFA (out/2022) é usado tanto para as partidas de treino (2018) quanto de teste (2022) — para 2018 isso é, na prática, ranking "do futuro". `dia16` mede o gap de correlação treino/teste por feature (as mais afetadas: `away_rank` e `rank_diff`) e retreina removendo-as: a acurácia média varia **+0.0pp** — o vazamento existe mas não é o principal motor do desempenho reportado.

### 9.5 O split fixo é robusto? Split invertido + 5-fold

Todo o pipeline usa um único split fixo (treino=2018, teste=2022). `dia17` testa o split invertido (treino=2022, teste=2018) e 5-fold estratificado sobre as 128 partidas combinadas:

| Modelo | Original (2018→2022) | Invertido (2022→2018) | 5-fold (média±desvio) |
|---|---|---|---|
| LogReg | 57.8% | 39.1% | 50.8% ± 6.5% |
| XGBoost | 50.0% | 50.0% | 54.7% ± 8.0% |
| LightGBM | 56.2% | 59.4% | 55.5% ± 2.9% |

O LogReg é frágil à direção do split (57.8%→39.1%); XGBoost/LightGBM são estáveis. O 5-fold (todos entre ~51-56%) é a estimativa mais confiável do que qualquer split único isoladamente.

### 9.6 Ensemble por média de probabilidades

Como nenhum modelo vence de forma robusta (9.1) e o "melhor" muda com o split (9.5), `dia18` testa a média das probabilidades calibradas dos 3 modelos: empata com o melhor individual no split original (57.8%) e é bem mais estável no invertido (51.6% vs 39.1% do LogReg puro), com o menor desvio padrão no 5-fold (3.9pp).

### 9.7 Regressão de Poisson nos gols — ataque ao zero-empates

O pipeline de classificação nunca prevê "Draw" nas 72 partidas de 2026. `dia19` modela os gols de cada lado via regressão de Poisson e deriva P(Home)/P(Draw)/P(Away) pela matriz de probabilidade de placares: o modelo dá um sinal de empate real (P(Draw) médio de 23.1% vs. confiança ~0% do classificador, "Draw" foi a 2ª opção mais provável em 42/64 jogos), mas a previsão pontual ainda não escolhe Draw — fenômeno conhecido de modelos de Poisson independentes subestimarem empates (correção clássica: Dixon-Coles, 1997).

### Conclusão

O resultado mais honesto deste projeto não é uma acurácia isolada, e sim: *com 64 amostras de teste, os modelos avaliados são estatisticamente equivalentes entre si e superiores ao acaso; o ranking FIFA carrega a maior parte do sinal disponível (mesmo descontando o vazamento temporal); a robustez ao split importa tanto quanto o pico de acurácia; e o ensemble e a modelagem de Poisson são respostas diretas, com evidência, às duas maiores fragilidades identificadas.*

### 9.8 Threshold tuning para Draw — testado e rejeitado

Um threshold "otimizado" de 0.10-0.20 na probabilidade de Draw parecia resolver o recall baixo no teste de 2022 (até 100% draw recall, mas só 23.4% accuracy). Validação **out-of-sample** contra o schedule real de 2026 (`notebooks/dia11_draw_recall_rps.py`) revelou o problema: o threshold=0.20 previu **45.8% de empates** (33/72 jogos) — mais que o dobro do esperado historicamente (~22%). O threshold escolhido no teste de 64 jogos não generalizou. **Decisão final: argmax puro, sem threshold**, aceitando draw recall baixo em troca de não arriscar overfitting fora de amostra.

### 9.9 Vazamento temporal no ranking FIFA

Revisão adicional identificou que todas as features de ranking (`home_rank`, `rank_diff`, etc.) usam **um único snapshot FIFA** (`fifa_ranking_2022-10-06.csv`) tanto para treino (2018) quanto para teste (2022) — as partidas de 2018 são treinadas com a força de cada seleção medida **~4 anos depois** do jogo. Evidência quantitativa: correlação rank_diff×resultado é **0.536 em 2018** (ranking "do futuro") vs. **0.346 em 2022** (ranking ~contemporâneo) — ~55% mais forte no ano vazado, consistente com vazamento inflando o sinal de treino. Não corrigível dentro do escopo oficial de dados da competição (não há ranking de época de 2018 nos arquivos autorizados) — registrado como limitação estrutural, não como bug.

### 9.10 Bug de nome de seleção corrompia ranking

O item "3 times sem ranking" (§8, Gaps) era na verdade um bug de join: `United States`, `Cape Verde` e `Bosnia-Herzegovina` não batiam com `USA`, `Cabo Verde` e `Bosnia and Herzegovina` nos CSVs de ranking FIFA, caindo em `NaN` → `fillna(0)` (rank #0 fantasma). Afetava também 4 das 64 partidas do teste de 2022 (Estados Unidos), corrompendo parte da acurácia "oficial" reportada. Corrigido com alias de nomes no ponto de join, sem dado externo ao escopo da competição. Efeito: no kernel Kaggle v3 (com H2H completo e seleção por RPS), as previsões 2026 ficaram **37 Casa/0 Fora/35 Empate** (o modelo final não prevê nenhum empate nos 72 jogos de fase de grupos).

### 9.6 Visualizações de incerteza

As afirmações estatísticas acima (§9.1) agora têm gráfico, não só tabela: `outputs/visualizations/07_incerteza_modelos.png` (comparação de modelos com IC 95%, barras de erro) e `08_confianca_previsoes_2026.png` (distribuição da probabilidade máxima das 72 previsões vs. o chute de 33.3%) — gerados por `notebooks/dia12_incerteza_viz.py`.